Importar las dependencias necesarias

In [2]:
import pandas as pd 
from sqlalchemy import create_engine

Importar el csv y verificar el DataFrame creado

In [3]:
ruta_csv = "../data/ventas_techventas.csv"
df = pd.read_csv(ruta_csv, encoding="latin1")
df.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
2,"1003,2024-01-10,C003,DataSolutions,Centro,Moni...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz


Crear la conexión a base de datos de SQlite y crear la tabla "ventas" 

In [4]:
engine = create_engine("sqlite:///../output/techventas.db")

In [5]:
df.to_sql(
    name="ventas",
    con=engine,
    if_exists="replace",
    index=False
)

60

# Query 1

Ejecutar la primera query SELECT DISTINCT: "Lista todos los productos únicos disponibles usando un alias de columna descriptivo "

In [6]:
query1 = """
SELECT DISTINCT producto AS ProductosUnicos
FROM ventas
"""

df1 = pd.read_sql(query1, engine)
df1

,ProductosUnicos
0,Laptop Pro 15
1,Mouse Inalambrico
2,NaN
3,Teclado Mecanico
4,SSD 1TB
5,Router WiFi 6


# Query 2

Ejecutar la segunda query WHERE + BETWEEN: "Filtra pedidos del primer trimestre (ene–mar 2024) con cantidad mayor a 5 unidades"

In [7]:
query2 = """ 
    SELECT *
    FROM ventas
    WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31'
    AND cantidad > 5;
    """
df2 = pd.read_sql(query2,engine)
df2

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
1,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
2,1006,2024-01-18,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20.0,95.0,0.00,Maria Torres
3,1008,2024-01-22,C006,NetPrime,Norte,Router WiFi 6,Redes,8.0,120.0,0.00,Luis Mora
4,1009,2024-01-25,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12.0,95.0,0.10,Maria Torres
5,1012,2024-02-05,C008,BetaSoft,Centro,Mouse Inalambrico,Perifericos,30.0,25.0,0.05,Maria Torres
6,1015,2024-02-12,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,25.0,95.0,0.10,Maria Torres
7,1018,2024-02-20,C002,Innovatek,Sur,SSD 1TB,Almacenamiento,18.0,95.0,0.00,Carlos Ruiz
8,1019,2024-02-22,C007,AlphaTech,Sur,Mouse Inalambrico,Perifericos,12.0,25.0,0.00,Maria Torres
9,1022,2024-03-04,C001,TechCorp SA,Norte,SSD 1TB,Almacenamiento,30.0,95.0,0.15,Carlos Ruiz


# Query 3

Ejecutar la tercera query GROUP BY + HAVING: Vendedores cuyo ingreso bruto total (cantidad × precio) supera $10,000 USD

In [8]:
query3 = """
    SELECT vendedor, 
           SUM(cantidad*precio_unitario) AS ingreso_total
    FROM ventas
    GROUP BY vendedor
    HAVING SUM(cantidad * precio_unitario) > 10000;
    """
df3 = pd.read_sql(query3, engine)
df3

,vendedor,ingreso_total
0,Ana Lï¿½ï¿,20810.0
1,Carlos Ruiz,21355.0
2,Luis Mora,19830.0
3,Maria Torres,11860.0


# Query 4

Ejecutar la cuarta query COUNT + SUM + AVG - Por categoría: total de pedidos, suma de unidades vendidas y precio unitario promedio.

In [9]:
query4 = """
    SELECT categoria,
           count(order_id) AS num_ventas,
           sum(cantidad) AS total_ventas,
           avg(precio_unitario) AS prom_precio
    FROM ventas
    GROUP BY categoria;
"""
df4 = pd.read_sql(query4,engine)
df4

,categoria,num_ventas,total_ventas,prom_precio
0,NaN,11,NaN,NaN
1,Almacenamiento,12,260.0,95.000000
2,Electronica,14,31.0,1139.285714
3,Perifericos,15,215.0,53.000000
4,Redes,8,39.0,120.000000


# Query 5

Definir función que se reutilizará para ejecutar consultas de creación de tablas e inserción de datos

In [22]:
from sqlalchemy import text

def run(sql):
    with engine.connect() as conn:
        result= conn.execute(text(sql))
        conn.commit()
        print("Filas afectadas:", result.rowcount)

Crear la tabla "vendedores"

In [23]:


run("""
        CREATE TABLE IF NOT EXISTS vendedores (
            vendedor TEXT PRIMARY KEY,
            zona TEXT,
            meta_mensual REAL
        )
    """)

Filas afectadas: -1


Verificar la creación correcta de las columnas de la tabla "vendedores"

In [24]:
pd.read_sql("SELECT * FROM vendedores", engine)

,vendedor,zona,meta_mensual
0,Ana López,Norte,8000.0
1,Carlos Ruiz,Sur,7500.0
2,Luis Mora,Norte,8000.0
3,Maria Torres,Centro,7000.0


Verificar si los datos se insertaron correctamente en la tabla

In [25]:
pd.read_sql("SELECT * FROM vendedores", engine)

,vendedor,zona,meta_mensual
0,Ana López,Norte,8000.0
1,Carlos Ruiz,Sur,7500.0
2,Luis Mora,Norte,8000.0
3,Maria Torres,Centro,7000.0


Antes de realizar el cálculo del cumplimiento es crucial corregir el nombre del vendedor "Ana López", por tanto, se realiza un update. 

In [26]:
run("""
        UPDATE ventas
        SET vendedor = 'Ana López'
        WHERE vendedor = 'Ana Lï¿½ï¿';
    """)


Filas afectadas: 0


In [27]:
pd.read_sql("""
SELECT DISTINCT vendedor
FROM ventas
ORDER BY vendedor
""", engine)

,vendedor
0,NaN
1,Ana López
2,Carlos Ruiz
3,Luis Mora
4,Maria Torres


In [28]:
pd.read_sql("""
SELECT vendedor
FROM vendedores
ORDER BY vendedor
""", engine)

,vendedor
0,Ana López
1,Carlos Ruiz
2,Luis Mora
3,Maria Torres


El porcentaje de cumplimiento se calcula de la siguiente manera: 
$\text{Cumplimiento (\%)} = \frac{\text{Ventas reales}}{\text{Meta}} \times 100$



$\text{Ventas reales}=(\text{cantidad} \times \text{precio\_unitario})\times (1-\text{descuento}) $

In [29]:
query5= """
SELECT
    m.vendedor,

    COALESCE(SUM(
        v.cantidad * v.precio_unitario * (1 - COALESCE(v.descuento, 0))
    ), 0) AS ingreso_total,

    m.meta_mensual,

    ROUND(
        (COALESCE(SUM(
            v.cantidad * v.precio_unitario * (1 - COALESCE(v.descuento, 0))
        ), 0) * 100.0) / m.meta_mensual,
        2
    ) AS porcentaje_cumplimiento

FROM vendedores m
LEFT JOIN ventas v
    ON m.vendedor = v.vendedor

GROUP BY
    m.vendedor,
    m.meta_mensual;
    """
pd.read_sql(query5,engine)

,vendedor,ingreso_total,meta_mensual,porcentaje_cumplimiento
0,Ana López,18249.0,8000.0,228.11
1,Carlos Ruiz,19255.5,7500.0,256.74
2,Luis Mora,18025.0,8000.0,225.31
3,Maria Torres,11436.0,7000.0,163.37
